In [1]:
'''CSV → data cleaning → type validation → join trades with market prices → valuation → P&L → exposure → risk'''

'CSV → data cleaning → type validation → join trades with market prices → valuation → P&L → exposure → risk'

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. load trading book

cwd = Path.cwd()
trade_file = cwd/ "energy_trading_data_example.csv"
df = pd.read_csv(trade_file)

# 2. Load market prices with timestamp

market_file = cwd/ "energy_market_price_example.csv"
df_market = pd.read_csv(market_file)

In [3]:
def to_numeric(df, columns):
    # to ensure numeric data
    df = df.copy()
    for col in columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("€", "", regex=False)
            .str.strip()
        )
        
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# errors="coerce" means that something like:
# "85.00"   → 85.0
# "20"      → 20.0
# "abc"     → NaN 

    df.info()
    return df

# def positions(df):
#     return df.groupby(['market','product','delivery_start','price_unit','unit'])['volume'].sum().reset_index().sort_values('volume', ascending=False)

def trades_market(df, df_market):
    return df.merge(df_market, on=['market','product','delivery_start', 'price_unit'], how='left')

def split_market(df_trades):
    df = df_trades.copy()

    market_mapping = {
        "SE3": ("Power", "SE3"),
        "DK1": ("Power", "DK1"),
        "DE-LU": ("Power", "DE-LU"),
        "FI": ("Power", "FI"),
        "TTF": ("Gas", "TTF"),
        "EUA": ("Carbon", "EUA"),
    }

    df["commodity"] = df["market"].map(
        lambda x: market_mapping[x][0]
    )

    df["location"] = df["market"].map(
        lambda x: market_mapping[x][1]
    )

    return df
    
def valuation(df_trades):
    # position_id, commodity, location, delivery, volume, entry_price, market_price, unrealized_p&L_€
    df = df_trades.copy()
    df['Unrealized_P&L_€'] = (df['market_price']-df['entry_price'])*df['volume']
    return df[['position_id', 'commodity', 'location', 'product','delivery_start', 'delivery_end',
               'volume', 'unit', 'entry_price', 'market_price', 'price_unit', 'Unrealized_P&L_€']]

def exposure_power(df_value):
    df = df_value.copy()
    zone_exposure = (
        df
        .query("commodity == 'Power' and unit == 'MWh'")
        .groupby(['location'])['volume']
        .agg(
            net_position="sum",
            gross_position=lambda x: x.abs().sum()
        )
        .reset_index()
    )
    tot_exposure = zone_exposure['net_position'].sum()
    print(f"Net power exposure is {tot_exposure} MWh")
    return zone_exposure.sort_values('gross_position', ascending=False)

def exposure_hourly(df_value):
    hourly_exposure = df_value.query("commodity == 'Power' and unit == 'MWh'").copy()
    hourly_exposure['delivery_start'] = pd.to_datetime(hourly_exposure['delivery_start'])
    hourly_exposure = (hourly_exposure
        .groupby('delivery_start')
        .agg(
            net_position=("volume", "sum"),
            gross_position=("volume",
                            lambda x: x.abs().sum()),
            number_of_positions=("position_id", "count")
        )
        .reset_index()
        .sort_values('delivery_start')
    )
    return hourly_exposure

def identify_spread_exposure(df_value):
    """
    Identify opposing power positions in different zones
    with the same delivery period.

    A spread is identified when one position is long
    and another is short for the same delivery_start.
    """

    power = df_value.query("commodity == 'Power' and unit == 'MWh'").copy()
    power["delivery_start"] = pd.to_datetime(power["delivery_start"])
    spreads = []

    for delivery_time, group in power.groupby("delivery_start"):
        long_positions = group[group["volume"] > 0]
        short_positions = group[group["volume"] < 0]
        for _, long_pos in long_positions.iterrows():
            for _, short_pos in short_positions.iterrows():
                if long_pos["location"] == short_pos["location"]:
                    continue

                spread_volume = min(
                    abs(long_pos["volume"]),
                    abs(short_pos["volume"])
                )

                spreads.append({
                    "delivery_start": delivery_time,
                    "long_location": long_pos["location"],
                    "short_location": short_pos["location"],
                    "spread_volume_mwh": spread_volume,
                    "long_position_id": long_pos["position_id"],
                    "short_position_id": short_pos["position_id"]
                })

    return pd.DataFrame(spreads)

def analyse_commodity_exposure(df_value):
    df = df_value.copy()
    commodity_exposure = (
        df
        .groupby(["commodity", "unit"])
        .agg(
            net_position=("volume", "sum"),
            gross_position=("volume",
                            lambda x: x.abs().sum()),
            number_of_positions=("position_id", "count")
        )
        .reset_index()
    )

    return commodity_exposure

def calculate_price_sensitivity(df_val, price_shock=10):
    """
    Simple Calculate P&L sensitivity to a uniform price shock.

    price_shock:
        EUR/MWh for Power/Gas
        EUR/tCO2 for Carbon
    a fixed absolute shock gives the same monetary sensitivity for the same position, 
    regardless of the starting price. 
    A percentage shock produces a different absolute price move 
    depending on the starting price
    """

    df = df_val.copy()
    df["price_shock"] = price_shock

    df["P&L_sensitivity_€"] = (
        df["volume"] * df["price_shock"]
    )

    sensitivity = (
        df
        .groupby(["commodity", "location","unit"])
        .agg(
            net_position=("volume", "sum"),
            price_shock=("price_shock", "first"),
            pnl_sensitivity=("P&L_sensitivity_€", "sum")
        )
        .reset_index()
    )

    return sensitivity

def stress_test(df_val, scenario):
    """
    Calculate stress P&L for a specified price scenario.

    scenario example:
        {
            "SE3": -20,
            "DK1": +20,
            "DE-LU": +20,
            "FI": -20
        }
    """

    df = df_val.query("commodity == 'Power' and unit == 'MWh'").copy()
    df["stress_shock"] = df["location"].map(scenario)

    # Positions without a defined shock receive zero
    df["stress_shock"] = df["stress_shock"].fillna(0)
    df["stress_pnl"] = (
        df["volume"] * df["stress_shock"]
    )

    result = (
        df
        .groupby("location")
        .agg(
            position=("volume", "sum"),
            shock=("stress_shock", "first"),
            stress_pnl=("stress_pnl", "sum")
        )
        .reset_index()
    )

    total_stress_pnl = result["stress_pnl"].sum()

    return result, total_stress_pnl

In [4]:
# 3. Clean & validate data
# 3.1 trading book
df = to_numeric(df, ['volume','entry_price'])
df

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   position_id      10 non-null     str  
 1   trade_timestamp  10 non-null     str  
 2   market           10 non-null     str  
 3   product          10 non-null     str  
 4   delivery_start   10 non-null     str  
 5   delivery_end     10 non-null     str  
 6   volume           10 non-null     int64
 7   unit             10 non-null     str  
 8   entry_price      10 non-null     int64
 9   price_unit       10 non-null     str  
dtypes: int64(2), str(8)
memory usage: 1.5 KB


,position_id,trade_timestamp,market,product,delivery_start,delivery_end,volume,unit,entry_price,price_unit
0,P001,10/7/2026 9:14,SE3,Day-Ahead,10/7/2026 18:00,19:00,20,MWh,85,EUR/MWh
1,P002,10/7/2026 9:18,DK1,Day-Ahead,10/7/2026 18:00,19:00,-20,MWh,175,EUR/MWh
2,P003,10/7/2026 9:32,DE-LU,Day-Ahead,10/7/2026 18:00,19:00,-15,MWh,180,EUR/MWh
3,P004,10/7/2026 10:05,SE3,Day-Ahead,10/7/2026 20:00,21:00,30,MWh,92,EUR/MWh
4,P005,10/7/2026 10:21,FI,Day-Ahead,10/8/2026 12:00,13:00,25,MWh,22,EUR/MWh
5,P006,10/7/2026 10:47,DK1,Day-Ahead,10/8/2026 19:00,20:00,-15,MWh,135,EUR/MWh
6,P007,10/7/2026 11:13,TTF,Month-Ahead,11/1/2026,12/1/2026,10,MW,31,EUR/MWh
7,P008,10/7/2026 11:41,DE-LU,Calendar-Year Future,1/1/2027,1/1/2028,-5,MW,82,EUR/MWh
8,P009,10/7/2026 12:03,SE3,Day-Ahead,10/8/2026 18:00,19:00,20,MWh,88,EUR/MWh
9,P010,10/7/2026 12:27,EUA,Dec-26 Future,12/1/2026,1/1/2027,1000,tCO₂,78,EUR/t


In [5]:
# 3.2 Market prices
df_market = to_numeric(df_market, ['market_price'])
df_market

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   market          10 non-null     str  
 1   product         10 non-null     str  
 2   delivery_start  10 non-null     str  
 3   market_price    10 non-null     int64
 4   price_unit      10 non-null     str  
 5   observed_at     10 non-null     str  
 6   source          10 non-null     str  
dtypes: int64(1), str(6)
memory usage: 1.2 KB


,market,product,delivery_start,market_price,price_unit,observed_at,source
0,SE3,Day-Ahead,10/7/2026 18:00,145,EUR/MWh,10/7/2026 15:00,Nord Pool
1,DK1,Day-Ahead,10/7/2026 18:00,205,EUR/MWh,10/7/2026 15:00,Nord Pool
2,DE-LU,Day-Ahead,10/7/2026 18:00,198,EUR/MWh,10/7/2026 15:00,EPEX
3,SE3,Day-Ahead,10/7/2026 20:00,132,EUR/MWh,10/7/2026 15:00,Nord Pool
4,FI,Day-Ahead,10/8/2026 12:00,18,EUR/MWh,10/7/2026 15:00,Nord Pool
5,DK1,Day-Ahead,10/8/2026 19:00,150,EUR/MWh,10/7/2026 15:00,Nord Pool
6,TTF,Month-Ahead,11/1/2026,33,EUR/MWh,10/7/2026 15:00,ICE
7,DE-LU,Calendar-Year Future,1/1/2027,79,EUR/MWh,10/7/2026 15:00,EEX
8,SE3,Day-Ahead,10/8/2026 18:00,110,EUR/MWh,10/7/2026 15:00,Nord Pool
9,EUA,Dec-26 Future,12/1/2026,80,EUR/t,10/7/2026 15:00,EEX


In [6]:
# 4. Calculate mart-to-market P&L

df_t = trades_market(df, df_market)
df_val = valuation(split_market(df_t))
df_val

,position_id,commodity,location,product,delivery_start,delivery_end,volume,unit,entry_price,market_price,price_unit,Unrealized_P&L_€
0,P001,Power,SE3,Day-Ahead,10/7/2026 18:00,19:00,20,MWh,85,145,EUR/MWh,1200
1,P002,Power,DK1,Day-Ahead,10/7/2026 18:00,19:00,-20,MWh,175,205,EUR/MWh,-600
2,P003,Power,DE-LU,Day-Ahead,10/7/2026 18:00,19:00,-15,MWh,180,198,EUR/MWh,-270
3,P004,Power,SE3,Day-Ahead,10/7/2026 20:00,21:00,30,MWh,92,132,EUR/MWh,1200
4,P005,Power,FI,Day-Ahead,10/8/2026 12:00,13:00,25,MWh,22,18,EUR/MWh,-100
5,P006,Power,DK1,Day-Ahead,10/8/2026 19:00,20:00,-15,MWh,135,150,EUR/MWh,-225
6,P007,Gas,TTF,Month-Ahead,11/1/2026,12/1/2026,10,MW,31,33,EUR/MWh,20
7,P008,Power,DE-LU,Calendar-Year Future,1/1/2027,1/1/2028,-5,MW,82,79,EUR/MWh,15
8,P009,Power,SE3,Day-Ahead,10/8/2026 18:00,19:00,20,MWh,88,110,EUR/MWh,440
9,P010,Carbon,EUA,Dec-26 Future,12/1/2026,1/1/2027,1000,tCO₂,78,80,EUR/t,2000


In [7]:
# 5. Calculate net power exposure in MWh
exposure_power(df_val)

Net power exposure is 45 MWh


,location,net_position,gross_position
3,SE3,70,70
1,DK1,-35,35
2,FI,25,25
0,DE-LU,-15,15


In [8]:
# 6. Exposure by delivery hour
exposure_hourly(df_val)

,delivery_start,net_position,gross_position,number_of_positions
0,2026-10-07 18:00:00,-15,55,3
1,2026-10-07 20:00:00,30,30,1
2,2026-10-08 12:00:00,25,25,1
3,2026-10-08 18:00:00,20,20,1
4,2026-10-08 19:00:00,-15,15,1


In [9]:
# 6.1 check delivery hour: 10/7/2026 18:00
df_val.query("commodity == 'Power' and unit == 'MWh' and delivery_start == '10/7/2026 18:00'")

,position_id,commodity,location,product,delivery_start,delivery_end,volume,unit,entry_price,market_price,price_unit,Unrealized_P&L_€
0,P001,Power,SE3,Day-Ahead,10/7/2026 18:00,19:00,20,MWh,85,145,EUR/MWh,1200
1,P002,Power,DK1,Day-Ahead,10/7/2026 18:00,19:00,-20,MWh,175,205,EUR/MWh,-600
2,P003,Power,DE-LU,Day-Ahead,10/7/2026 18:00,19:00,-15,MWh,180,198,EUR/MWh,-270


In [10]:
# 7. Identify spread exposure
print("Two possible cross-zonal spreads at the same delivery hour.")
print("The current function is useful for detecting potential pairwise spreads, but not for allocating the portfolio's exposure")
identify_spread_exposure(df_val)

Two possible cross-zonal spreads at the same delivery hour.
The current function is useful for detecting potential pairwise spreads, but not for allocating the portfolio's exposure


,delivery_start,long_location,short_location,spread_volume_mwh,long_position_id,short_position_id
0,2026-10-07 18:00:00,SE3,DK1,20,P001,P002
1,2026-10-07 18:00:00,SE3,DE-LU,15,P001,P003


In [11]:
# 8. Exposure by commodity
analyse_commodity_exposure(df_val)

,commodity,unit,net_position,gross_position,number_of_positions
0,Carbon,tCO₂,1000,1000,1
1,Gas,MW,10,10,1
2,Power,MW,-5,5,1
3,Power,MWh,45,145,7


In [12]:
# 9. Calculate monetary sensitivity: P&L sensitivity to price movement (price shock): defualt a uniform shock of +€10/MWh
# Price sensitivity tells how much would the portfolio gain or lose if a particular market price moved by a specified amount
calculate_price_sensitivity(df_val, price_shock=10)

,commodity,location,unit,net_position,price_shock,pnl_sensitivity
0,Carbon,EUA,tCO₂,1000,10,10000
1,Gas,TTF,MW,10,10,100
2,Power,DE-LU,MW,-5,10,-50
3,Power,DE-LU,MWh,-15,10,-150
4,Power,DK1,MWh,-35,10,-350
5,Power,FI,MWh,25,10,250
6,Power,SE3,MWh,70,10,700


In [13]:
# 10. Calculate Stress Testing on day-ahead power maket
# Stress testing tells how much could the portfolio gain or lose under a defined, potentially adverse market scenario 
# involving one or more market variables

# 10.1 single scenario
scenario = {
    "SE3": -20,
    "DK1": +20,
    "DE-LU": +20,
    "FI": -20
}

'''
A plausible narrative is stronger renewable or hydro availability in 
Sweden and Finland coinciding with tighter supply conditions in Denmark and Germany.
'''

stress_detail, total_stress_pnl = stress_test(
    df_val,
    scenario
)

print(stress_detail)
print(f"Total stress P&L: €{total_stress_pnl:,.0f}")


  location  position  shock  stress_pnl
0    DE-LU       -15     20        -300
1      DK1       -35     20        -700
2       FI        25    -20        -500
3      SE3        70    -20       -1400
Total stress P&L: €-2,900


In [14]:
# 10.2 multiple scenario

def run_scenarios(df_val, scenarios):

    results = []

    for scenario_name, scenario in scenarios.items():

        _, stress_pnl = stress_test(
            df_val,
            scenario
        )

        results.append({
            "scenario": scenario_name,
            "stress_pnl": stress_pnl
        })

    return pd.DataFrame(results)
    
scenarios = {
    "Nordic adverse": {
        "SE3": -20,
        "DK1": +20,
        "FI": -20,
        "DE-LU": 0
    },

    "European price spike": {
        "SE3": +50,
        "DK1": +50,
        "FI": +50,
        "DE-LU": +50
    },

    "European price crash": {
        "SE3": -50,
        "DK1": -50,
        "FI": -50,
        "DE-LU": -50
    },

    "SE3-DK1 convergence": {
        "SE3": -20,
        "DK1": -20,
        "FI": 0,
        "DE-LU": 0
    }
}

'''
Nordic adverse: Nordic power prices weaken in Sweden and Finland, while Denmark becomes more expensive relative to them.
Possible drivers: Higher wind or hydro availability, lower demand, or improved supply in Sweden and Finland could depress prices. 
Meanwhile, weaker wind generation in Denmark, stronger demand, or constraints on cross-border transmission could push Danish prices higher.

European price spike: Electricity prices rise simultaneously across all four bidding zones.
Possible drivers:
- A widespread drop in wind and solar generation.
- Cold weather increasing heating demand.
- Unexpected outages at major power plants.
- Higher gas prices increasing the marginal cost of gas-fired generation.
- Tighter regional supply and limited reserve capacity.

European price crash: Electricity prices fall sharply across all four zones.
Possible drivers:
- Very high wind and solar generation.
- Low demand due to mild weather or reduced industrial consumption.
- Nuclear or other inflexible generation remaining online during periods of excess supply.
- Transmission capacity allowing surplus electricity to spread across neighbouring markets.

SE3–DK1 convergence: Swedish SE3 and Danish DK1 prices both fall by €20/MWh, while Finnish and German prices remain unchanged
Possible drivers: Lower demand, stronger wind generation, or a broader decline in Nordic power prices could lower both zones. 
The absence of a change in FI and DE-LU is a simplifying assumption used to isolate the effect on SE3 and DK1.
In this case both SE3 and DK1 move in the same direction, so it does not directly widen or narrow their price spread.
'''

stress_results = run_scenarios(
    df_val,
    scenarios
)

print(stress_results)

               scenario  stress_pnl
0        Nordic adverse       -2600
1  European price spike        2250
2  European price crash       -2250
3   SE3-DK1 convergence        -700


In [30]:
def portfolio_report(valuation):

    print("=" * 60)
    print("EUROPEAN ENERGY PORTFOLIO RISK REPORT")
    print("=" * 60)

    # --------------------------------------------------
    # 1. P&L
    # --------------------------------------------------

    total_pnl = valuation["Unrealized_P&L_€"].sum()

    print("\nP&L")
    print("-" * 60)
    print(f"Unrealised P&L: €{total_pnl:,.2f}")

    # --------------------------------------------------
    # 2. Power exposure
    # --------------------------------------------------

    power = valuation[
        (valuation["commodity"] == "Power") &
        (valuation["unit"] == "MWh")
    ]

    net_power = power["volume"].sum()
    gross_power = power["volume"].abs().sum()

    print("\nPOWER EXPOSURE")
    print("-" * 60)
    print(f"Net position:   {net_power:+,.0f} MWh")
    print(f"Gross position: {gross_power:,.0f} MWh")

    # --------------------------------------------------
    # 3. Exposure by zone
    # --------------------------------------------------

    zone_exposure = (
        power
        .groupby("location")["volume"]
        .agg(
            net="sum",
            gross=lambda x: x.abs().sum()
        )
    )

    print("\nPOWER EXPOSURE BY ZONE")
    print("-" * 60)
    print(zone_exposure)

    # --------------------------------------------------
    # 4. Commodity exposure
    # --------------------------------------------------

    commodity_exposure = (
        valuation
        .groupby(["commodity", "unit"])["volume"]
        .agg(
            net="sum",
            gross=lambda x: x.abs().sum()
        )
    )

    print("\nCOMMODITY EXPOSURE")
    print("-" * 60)
    print(commodity_exposure)

    # --------------------------------------------------
    # 5. Delivery-hour exposure
    # --------------------------------------------------

    hourly_exposure = (
        power
        .groupby("delivery_start")["volume"]
        .agg(
            net="sum",
            gross=lambda x: x.abs().sum()
        )
    )

    print("\nEXPOSURE BY DELIVERY HOUR")
    print("-" * 60)
    print(hourly_exposure)

    valuation_timestamp = pd.Timestamp.now(tz="Europe/Stockholm")
    print(
    "\nValuation timestamp:",
    valuation_timestamp.strftime("%Y-%m-%d %H:%M:%S %Z")
    )


    

    return {
        "total_pnl": total_pnl,
        "zone_exposure": zone_exposure,
        "commodity_exposure": commodity_exposure,
        "hourly_exposure": hourly_exposure
    }

import pandas as pd


def get_price_timestamps(market_prices):
    df = market_prices.copy()

    df["observed_at"] = pd.to_datetime(
        df["observed_at"],
        errors="coerce",
        utc=True
    )

    # Group by commodity and market/location
    timestamps = (
        df.groupby(["product", "market"], dropna=False)
        .agg(
            latest_price_timestamp=("observed_at", "max"),
            price_records=("observed_at", "count")
        )
        .reset_index()
        .sort_values(["product", "market"])
    )

    # Convert timestamps to Central European local time for reporting.
    timestamps["latest_price_timestamp"] = (
        timestamps["latest_price_timestamp"]
        .dt.tz_convert("Europe/Stockholm")
        .dt.strftime("%Y-%m-%d %H:%M:%S %Z")
    )

    print("\nLATEST MARKET PRICE TIMESTAMPS")
    print(timestamps.to_string(index=False))

    return timestamps


In [31]:
# 11. Portfolio report
report = portfolio_report(df_val)

EUROPEAN ENERGY PORTFOLIO RISK REPORT

P&L
------------------------------------------------------------
Unrealised P&L: €3,680.00

POWER EXPOSURE
------------------------------------------------------------
Net position:   +45 MWh
Gross position: 145 MWh

POWER EXPOSURE BY ZONE
------------------------------------------------------------
          net  gross
location            
DE-LU     -15     15
DK1       -35     35
FI         25     25
SE3        70     70

COMMODITY EXPOSURE
------------------------------------------------------------
                 net  gross
commodity unit             
Carbon    tCO₂  1000   1000
Gas       MW      10     10
Power     MW      -5      5
          MWh     45    145

EXPOSURE BY DELIVERY HOUR
------------------------------------------------------------
                 net  gross
delivery_start             
10/7/2026 18:00  -15     55
10/7/2026 20:00   30     30
10/8/2026 12:00   25     25
10/8/2026 18:00   20     20
10/8/2026 19:00  -15     15



In [32]:
time_report = get_price_timestamps(df_market)


LATEST MARKET PRICE TIMESTAMPS
             product market   latest_price_timestamp  price_records
Calendar-Year Future  DE-LU 2026-10-07 17:00:00 CEST              1
           Day-Ahead  DE-LU 2026-10-07 17:00:00 CEST              1
           Day-Ahead    DK1 2026-10-07 17:00:00 CEST              2
           Day-Ahead     FI 2026-10-07 17:00:00 CEST              1
           Day-Ahead    SE3 2026-10-07 17:00:00 CEST              3
       Dec-26 Future    EUA 2026-10-07 17:00:00 CEST              1
         Month-Ahead    TTF 2026-10-07 17:00:00 CEST              1
